In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
# from pyspark.sql.functions import col
# from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup


# class TrainingSetBuilder:

#     def __init__(self, spark, labels_table, feature_store_table, primary_key, label_column):
#         self.spark = spark
#         self.labels_table = labels_table
#         self.feature_store_table = feature_store_table
#         self.primary_key = primary_key
#         self.label_column = label_column
#         self.fe = FeatureEngineeringClient()
#         self.training_set = None

#     def build(self):
#         base_df = self.spark.table(self.labels_table)

#         feature_lookups = [
#             FeatureLookup(
#                 table_name=self.feature_store_table,
#                 lookup_key=self.primary_key
#             )
#         ]

#         self.training_set = self.fe.create_training_set(
#             df=base_df,
#             feature_lookups=feature_lookups,
#             label=self.label_column,
#             exclude_columns=[self.primary_key]
#         )

#         all_data_df = self.training_set.load_df()
#         return all_data_df


# class TimeSeriesSplitter:

#     def __init__(self, spark, date_column):
#         self.spark = spark
#         self.date_column = date_column

#     def split(self, df, split_ratio=0.8):
#         distinct_dates = [
#             row[self.date_column]
#             for row in df.select(self.date_column).distinct().orderBy(self.date_column).collect()
#         ]

#         cutoff_index = int(len(distinct_dates) * split_ratio) - 1
#         cutoff_date = distinct_dates[cutoff_index]

#         early_df = df.filter(col(self.date_column) <= cutoff_date)
#         late_df = df.filter(col(self.date_column) > cutoff_date)

#         return early_df, late_df, cutoff_date


# # ---------------------------
# # Notebook usage
# # ---------------------------

# builder = TrainingSetBuilder(
#     spark=spark,
#     labels_table="oil_gas_mlops.gold.labels_supply_chain",
#     feature_store_table="oil_gas_mlops.gold.feature_store_supply_chain",
#     primary_key="feature_store_id",
#     label_column="total_demand_quantity"
# )
# all_data_df = builder.build()
# print("Total rows: " + str(all_data_df.count()))

# splitter = TimeSeriesSplitter(spark, date_column="transaction_date")
# train_val_df, test_df, split_1_cutoff = splitter.split(all_data_df, split_ratio=0.8)
# print("Train+Val rows: " + str(train_val_df.count()) + " | Test rows: " + str(test_df.count()))

# train_df, validation_df, split_2_cutoff = splitter.split(train_val_df, split_ratio=0.8)
# print("Train rows: " + str(train_df.count()) + " | Validation rows: " + str(validation_df.count()))

In [0]:
from pyspark.sql.functions import col
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup


# ============================================================
# 1. TRAINING SET BUILDER (joins labels with features)
# ============================================================

class TrainingSetBuilder:

    def __init__(self, spark, labels_table, feature_store_table, key_columns, label_column):
        self.spark = spark
        self.labels_table = labels_table
        self.feature_store_table = feature_store_table
        self.key_columns = key_columns
        self.label_column = label_column
        self.fe = FeatureEngineeringClient()
        self.training_set = None

    def build(self):
        # Base table: key columns + label
        base_df = self.spark.table(self.labels_table)

        feature_store_df = self.spark.table(self.feature_store_table)
        feature_names = [
        c for c in feature_store_df.columns
        if c not in self.key_columns 
        ]

        # Look up features by the composite key
        feature_lookups = [
            FeatureLookup(
                table_name=self.feature_store_table,
                lookup_key=self.key_columns,
                feature_names=feature_names
            )
        ]

        self.training_set = self.fe.create_training_set(
            df=base_df,
            feature_lookups=feature_lookups,
            label=self.label_column
            
        )

        all_data_df = self.training_set.load_df()
        return all_data_df


# ============================================================
# 2. TIME SERIES SPLITTER (chronological split)
# ============================================================

class TimeSeriesSplitter:

    def __init__(self, spark, date_column):
        self.spark = spark
        self.date_column = date_column

    def split(self, df, split_ratio=0.8):
        distinct_dates = [
            row[self.date_column]
            for row in df.select(self.date_column).distinct().orderBy(self.date_column).collect()
        ]

        cutoff_index = int(len(distinct_dates) * split_ratio) - 1
        cutoff_date = distinct_dates[cutoff_index]

        early_df = df.filter(col(self.date_column) <= cutoff_date)
        late_df = df.filter(col(self.date_column) > cutoff_date)

        return early_df, late_df, cutoff_date


# ============================================================
# 3. RUN
# ============================================================

key_columns = ["transaction_date", "product_name", "destination_city"]

# Step 1: Join labels + features -> 100% of the data
builder = TrainingSetBuilder(
    spark=spark,
    labels_table="oil_gas_mlops.gold.labels_supply_chain",
    feature_store_table="oil_gas_mlops.gold.feature_store_supply_chain",
    key_columns=key_columns,
    label_column="total_demand_quantity"
)
all_data_df = builder.build()
print("Total rows: " + str(all_data_df.count()))

# Step 2: Split 100% -> train_val (80%) + test (20%)
splitter = TimeSeriesSplitter(spark, date_column="transaction_date")
train_val_df, test_df, split_1_cutoff = splitter.split(all_data_df, split_ratio=0.8)
print("Train+Val rows: " + str(train_val_df.count()) + " | Test rows: " + str(test_df.count()))

# Step 3: Split train_val -> train (80%) + validation (20%)
train_df, validation_df, split_2_cutoff = splitter.split(train_val_df, split_ratio=0.8)
print("Train rows: " + str(train_df.count()) + " | Validation rows: " + str(validation_df.count()))